In [89]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

In [90]:
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from pathlib import Path
project_root=Path.cwd().parent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq

In [91]:
# For Api Rate limiting
import logging
from tenacity import (
     retry,  # Decorator that wraps a function with retry logic
    stop_after_attempt,  # Stop after N total attempts
    wait_exponential,  # Wait 1s, 2s, 4s, 8s between retries
    before_sleep_log, 

)


logging.basicConfig(
    level=logging.INFO,
    filename=project_root/"log"/"sql_agent.log",
    filemode="w",

)

logger = logging.getLogger(__name__)
attempt_counter={"n":0}


In [92]:
# Define the location of the SQLite database
db_loc = project_root/'Database'/'ecomm.db'

# Create a SQLDatabase instance from the SQLite database URI
db = SQLDatabase.from_uri(f"sqlite:///{db_loc}")

# Retrieve the schema information of the database tables
database_schema = db.get_table_info()

In [93]:
print(database_schema)


CREATE TABLE "Customers" (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone INTEGER NOT NULL, 
	address TEXT NOT NULL, 
	PRIMARY KEY (customer_id)
)

/*
3 rows from Customers table:
customer_id	first_name	last_name	email	phone	address
1	David	Taylor	david.taylor@yahoo.com	8010878513	1130 Birch Blvd, Chicago, AZ, 81011
2	Emily	Davis	emily.davis@outlook.com	6281972072	7897 Willow Ln, Chicago, CA, 55316
3	Michael	Smith	michael.smith@outlook.com	7579377501	2834 Oak St, Philadelphia, IL, 10112
*/


CREATE TABLE "Invoices" (
	order_id INTEGER NOT NULL, 
	invoice_date TEXT NOT NULL, 
	amount REAL NOT NULL, 
	invoice_url TEXT NOT NULL, 
	FOREIGN KEY(order_id) REFERENCES "Orders" (order_id)
)

/*
3 rows from Invoices table:
order_id	invoice_date	amount	invoice_url
2	2024-11-17 23:31:11	432.92	https://example.com/invoice/2
3	2024-11-25 16:20:51	473.85	https://example.com/invoice/3
4	2024-11-24 17:47:29	117.39	https://example.com/invoice

In [94]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key=os.getenv("api_key")

In [95]:

config_file = project_root / "configs" / "sql_agent.json"


In [96]:
# Load the JSON configuration
with open(config_file, "r", encoding="utf-8") as f:
    config_data = json.load(f)
    
# Extract the string path and convert it back into a pathlib.Path
raw_path = config_data["sql_agent"]["v1"]["prompt_path"]

prompt_path = project_root/raw_path  # or Path(raw_path) if it's absolute
model=config_data["sql_agent"]["v1"]["config"]["model"]
temprature=config_data["sql_agent"]["v1"]["config"]["temperature"]


In [97]:
with open(prompt_path,"r",encoding="utf-8") as file : 
    sql_prompt=file.read()

formatted_system_prompt = sql_prompt.format(database_schema=database_schema) 
# Create a full prompt template for the agent using the system message and placeholders
full_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", sql_prompt),
        ("human", '{input}'),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

In [98]:
llm=ChatGroq(
    model=model,
    temperature=temprature,
    api_key=api_key 
)

In [99]:
# Create the SQL agent using the AzureChatOpenAI model, database, and prompt template
sqlite_agent = create_sql_agent(
    llm=llm,
    db=db,
    prompt=full_prompt,
    agent_type="openai-tools",
    agent_executor_kwargs={'handle_parsing_errors': True},
    max_iterations=5,
    verbose=True
)



In [100]:

@tool
def sql_tool(user_input):
    """
    Executes a SQL query using the sqlite_agent and returns the result.

    Args:
        user_input (str): a natural language query string explaining what information is required while also providing the necessary details to get the information.

    Returns:
        str: The result of the SQL query execution. If an error occurs, the exception is returned as a string.
    """
    try:
        # Invoke the sqlite_agent with the user input (SQL query)
        response = sqlite_agent.invoke(user_input)

        # Extract the output from the response
        prediction = response['output']

    except Exception as e:
        # If an exception occurs, capture the exception message
        prediction = e

    # Return the result or the exception message
    return prediction

In [101]:
# # Test the sql_tool function with a sample user input
user_input = "Did Michael	Smith place an order?"
answer = sql_tool.invoke(user_input)

@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1,min=1,max=10),
    before_sleep=before_sleep_log(logger,logging.WARNING)
)

def run_sql_agent(user_query: str):
    # 2. Put your agent executor invoke code inside this function
    answer = sql_tool.invoke(user_input)
    return answer
print(run_sql_agent(user_input))



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_query` with `{'query': "SELECT COUNT(*) FROM Customers c JOIN Orders o ON c.customer_id = o.customer_id WHERE c.first_name = 'Michael' AND c.last_name = 'Smith'"}`


[(2,)]Yes, Michael Smith placed 2 orders.

> Finished chain.


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_query` with `{'query': "SELECT COUNT(*) FROM Customers c JOIN Orders o ON c.customer_id = o.customer_id WHERE c.first_name = 'Michael' AND c.last_name = 'Smith'"}`


[(2,)]Yes, Michael Smith placed 2 orders.

> Finished chain.
Yes, Michael Smith placed 2 orders.
